# [VD-05] HWPX 기초부터 2026 초기창업패키지 사업계획서 완성까지
### 실무형 한글(OWPML) 자동 생성 및 무결성 검증 마스터 클래스
---
- **기초 기술 출처**: 이현구 교수 OWPML 강의
- **고도화 및 실무 구축**: (AX)창업기술 대표 이한규 (VD Master Series)
- **핵심 학습 목표**:
  1. HWPX (OWPML) 파일 구조의 3대 필수 원칙 이해 (mimetype 압축 금지, OCF 컨테이너, HPF 패키징)
  2. 텍스트 치환 시 발생하는 글씨 겹침 버그(`linesegarray`)의 근본 원인 및 100% 치료법 실습
  3. 붕어빵 틀 기법(Mold Pattern)을 통한 동적 표(`<hp:tbl>`) 생성
  4. 2026년도 초기창업패키지 PSST 표준 사업계획서 HWPX 원클릭 자동 생성

In [ ]:
# 1. 환경 설정 및 모듈 경로 로드
import os
import sys
from pathlib import Path

# 현재 폴더 및 상위 scripts 폴더를 sys.path에 추가
base_dir = Path.cwd().parent if Path.cwd().name == 'jupyternotebook' else Path.cwd()
scripts_dir = base_dir / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print(f'[작업 기준 디렉토리]: {base_dir}')
print(f'[스크립트 경로]: {scripts_dir}')

## Step 1. HWPX 내부 구조 심층 해부
HWPX 파일은 실제로는 ZIP 압축 포맷입니다. 단, 한컴오피스가 정상 인식하기 위한 **3대 불변 규칙**이 있습니다:
1. `mimetype` 파일이 압축 파일의 **가장 첫 번째 항목(First Entry)**이어야 함
2. `mimetype` 파일은 **압축하지 않은 원형(ZIP_STORED, 0% compression)**으로 기록되어야 함
3. `header.xml`의 태그 개수 카운트(`charPrCnt`, `borderFillCnt` 등)가 실제 태그 수와 정확히 일치해야 함

In [ ]:
import zipfile

sample_hwpx = base_dir / 'templates' / '01_초기창업패키지_사업계획서_표준양식.hwpx'
print('=== HWPX 내부 파일 아카이브 목록 ===')
with zipfile.ZipFile(sample_hwpx, 'r') as z:
    for idx, info in enumerate(z.infolist()):
        comp = '무압축(STORED)' if info.compress_type == zipfile.ZIP_STORED else '압축(DEFLATED)'
        print(f'{idx+1:02d}. {info.filename:<30} | 크기: {info.file_size:6d}B | 방식: {comp}')

## Step 2. 글씨 겹침 현상과 `linesegarray` (레이아웃 캐시) 제거
파이썬으로 텍스트를 수정하면 글자 길이가 달라지지만, 한글이 기존에 계산해 둔 줄바꿈 캐시(`<hp:linesegarray>`)가 남아 있으면 글자가 겹쳐서 출력되는 치명적인 결함이 발생합니다.
`clear_layout_cache.py`를 통해 이 캐시를 제거하면, 한글 프로그램이 문서를 열 때 정확한 줄바꿈과 표 높이를 자동 재계산합니다.

In [ ]:
from clear_layout_cache import count_file, process

tmpl_file = base_dir / 'templates' / '02_공공기관_기본보고서_양식.hwpx'
print(f'공공기관 템플릿 내 기존 캐시 개수: {count_file(str(tmpl_file))}개')
print('-> 파이썬 치환 시 캐시를 삭제해야만 한글에서 글씨 겹침이 방지됩니다.')

## Step 3. 동적 표(Table) 생성: 붕어빵 틀 기법
데이터 행 수가 가변적인 표를 만들 때, 헤더 서식과 셀 서식을 완벽히 준수하는 `<hp:tbl>`을 만듭니다.

In [ ]:
from table_builder import build_table_xml

headers = ['비목', '산출근거', '금액(원)', '비율(%)']
rows = [
    ['인건비', 'AI 엔지니어 2인 6개월', '40,000,000', '40.0%'],
    ['재료비', '클라우드 GPU 인프라 서버비', '30,000,000', '30.0%'],
    ['외주용역비', '시니어 UX/UI 사용성 검증', '20,000,000', '20.0%'],
    ['지식재산권', '특허 출원 2건 및 상표권', '10,000,000', '10.0%']
]

tbl_xml = build_table_xml(headers, rows)
print(f'생성된 표 XML 길이: {len(tbl_xml)} 글자')
print(tbl_xml[:350] + '...')

## Step 4. 2026 초기창업패키지 사업계획서(PSST) 원클릭 빌드
`psst_builder` 모듈을 사용하여 4대 영역(Problem, Solution, Scale-up, Team)을 갖춘 완전한 HWPX 문서를 생성합니다.

In [ ]:
from psst_builder import build_psst_hwpx
from result_manager import get_next_hwpx_filename

output_hwpx = get_next_hwpx_filename(base_dir / 'result', '주피터_초기창업패키지_실습')
build_psst_hwpx(
    output_path=output_hwpx,
    doc_title='2026년도 초기창업패키지 사업계획서',
    item_name='생성형 AI 기반 시니어 디지털 돌봄 플랫폼',
    ceo_name='이한규 대표',
    category='인공지능 / 실버테크',
    gov_fund='80,000,000원',
    self_fund='20,000,000원',
    period='2026.05 ~ 2027.02',
    problem_text='- 초고령화 시대 시니어 정보 소외 문제 해결\n- 복잡한 모바일 인터페이스 극복 필요',
    solution_text='- 음성 기반 1:1 대화형 스마트 가이드 시스템 개발\n- 직관적 화면 시뮬레이터 제공',
    scaleup_text='- 전국 250개 노인복지관 및 평생학습관 B2G 공급\n- 민간 요양 플랫폼 연계 B2B 확장',
    team_text='- 대표자 AI 사업화 15년 경력\n- 석박사급 AI 개발진 보유'
)
print(f'[성공] 사업계획서가 생성되었습니다: {output_hwpx.name}')

## Step 5. HWPX 무결성 및 규격 최종 검증 (`verify_hwpx`)
생성된 파일이 한컴오피스에서 완벽하게 열릴 수 있는지 사전 진단합니다.

In [ ]:
from verify_hwpx import check

is_valid = check(str(output_hwpx))
print(f'최종 검증 결과: {"PASS (합격)" if is_valid else "FAIL (불합격)"}')